# Analysis

This notebook covers the analysis methods available in *PedPy*.
Before diving into the analysis, we need to set up the walkable area, measurement areas, and load trajectory data.
For more details on how to set up your analysis environment, see the {doc}`Measurement Setup <measurement_setup>` notebook. 

## Measurement setup

For showing the capabilities of *PedPy* we are a using experimental data of bottleneck experiments conducted in 2018 at the University of Wuppertal. 
The data for this experiment is available {download}`here <demo-data/bottleneck/040_c_56_h-.txt>`, which belongs to this [experimental series](https://doi.org/10.34735/ped.2018.1) and is part of the publication ["Crowds in front of bottlenecks at entrances from the perspective of physics and social psychology"](https://doi.org/10.1098/rsif.2019.0871).
We are re-using the measurement setup of the [Getting Started Guide](getting_started) and load the trajectories from the provided txt-file. 

This results in the following measurement setup:

In [ ]:
import pathlib
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely

warnings.filterwarnings("ignore")

from pedpy import (
    MeasurementArea,
    MeasurementLine,
    TrajectoryUnit,
    WalkableArea,
    load_trajectory,
    plot_measurement_setup,
)

walkable_area = WalkableArea(
    [
        (3.5, -2),
        (3.5, 8),
        (-3.5, 8),
        (-3.5, -2),
    ],
    obstacles=[
        [
            (-0.7, -1.1),
            (-0.25, -1.1),
            (-0.25, -0.15),
            (-0.4, 0.0),
            (-2.8, 0.0),
            (-2.8, 6.7),
            (-3.05, 6.7),
            (-3.05, -0.3),
            (-0.7, -0.3),
            (-0.7, -1.0),
        ],
        [
            (0.25, -1.1),
            (0.7, -1.1),
            (0.7, -0.3),
            (3.05, -0.3),
            (3.05, 6.7),
            (2.8, 6.7),
            (2.8, 0.0),
            (0.4, 0.0),
            (0.25, -0.15),
            (0.25, -1.1),
        ],
    ],
)

measurement_area = MeasurementArea([(-0.4, 0.5), (0.4, 0.5), (0.4, 1.3), (-0.4, 1.3)])
measurement_line = MeasurementLine([(0.4, 0), (-0.4, 0)])

traj = load_trajectory(
    trajectory_file=pathlib.Path("demo-data/bottleneck/040_c_56_h-.txt"),
    default_unit=TrajectoryUnit.METER,
)

plot_measurement_setup(
    traj=traj,
    walkable_area=walkable_area,
    measurement_lines=[measurement_line],
    ml_width=2,
    measurement_areas=[measurement_area],
    ma_line_width=2,
    ma_alpha=0.2,
).set_aspect("equal")
plt.show()

## Density

Density is a fundamental metric in pedestrian dynamics.
As it indicated how much space is accessible to each pedestrian within a specific area.
High density can lead to reduced walking speeds, increased congestion, and even potential safety hazards.

### Classic density

The classic approach to calculate the density $\rho_{classic}(t)$ at a time $t$, is to count the number of pedestrians ($N(t)$) inside a specific space ($M$) and divide it by the area of that space ($A(M)$).

$$
\rho_{classic}(t) = {N(t) \over A(M)}
$$

In *PedPy* this can be computed with:


In [ ]:
from pedpy import compute_classic_density

classic_density = compute_classic_density(traj_data=traj, measurement_area=measurement_area)

The resulting time-series can be seen below:

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_density

plot_density(density=classic_density, title="Classic density")
plt.show()

(voronoi_density)=
 ### Voronoi density

Another approach for calculating the density is to compute the [Voronoi tesselation](https://en.wikipedia.org/wiki/Voronoi_diagram) of the pedestrians positions at a given time $t$, the resulting Voronoi polygons ($V$) directly relate to the individual's density.
For a pedestrian $i$ the individual density is defined as:

$$
\rho_i(t) = {1 \over A(V_i(t))}
$$


#### Compute individual Voronoi Polygons

The first step for computing the Voronoi density, is to compute the individual's Voronoi polygon.
As these polygons may become infinite for pedestrians at the edge of the crowd, these polygons are restricted by the {class}`walkable area <geometry.WalkableArea>`.
This cutting at the boundaries can lead to split Voronoi polygons. For each of the split polygons it is checked, in which the pedestrian is located. 
This polygon then is assigned.

:::{important}
As these Voronoi polygons work on the Euclidean distance, some unexpected artifacts may occur on non-convex {class}`walkable areas <geometry.WalkableArea>`. Please keep that in mind! How that may look like, you can see in the plots later in this guide.
:::

##### Without cut-off

The computation of the individual polygons can be done from the {class}`trajectory data<trajectory_data.TrajectoryData>` and {class}`walkable area <geometry.WalkableArea>` with:

In [ ]:
from pedpy import compute_individual_voronoi_polygons

individual = compute_individual_voronoi_polygons(traj_data=traj, walkable_area=walkable_area)

##### With cut-off

When having a large {class}`walkable area <geometry.WalkableArea>` or widely spread pedestrians the Voronoi polygons may become quite large.
In *PedPy* it is possible to restrict the size of the computed Polygons.
This can be done by defining a {class}`cut off <method_utils.CutOff>`, which is essentially an approximated circle which gives the maximum extension of a single Voronoi polygon.
For the creation of the {class}`cut off <method_utils.CutOff>`, we need to define how accurate we want to approximate the circle, the differences can be seen below:

```{eval-rst}
.. figure:: /images/voronoi_cutoff_differences.svg
    :align: center
```

Now, with that {class}`cut off <method_utils.CutOff>` the computation of the individual polygons becomes:

In [ ]:
from pedpy import Cutoff, compute_individual_voronoi_polygons

individual_cutoff = compute_individual_voronoi_polygons(
    traj_data=traj,
    walkable_area=walkable_area,
    cut_off=Cutoff(radius=1.0, quad_segments=3),
)

##### Comparison

To get a better impression what the differences between the Voronoi polygons with and without the {class}`cut off <method_utils.CutOff>` are, take a look at the plot below:

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

from pedpy import DENSITY_COL, FRAME_COL, ID_COL, plot_voronoi_cells

frame = 600

fig = plt.figure(f"frame = {frame}")
fig.suptitle(f"frame = {frame}")

ax1 = fig.add_subplot(121, aspect="equal")
ax1.set_title("w/o cutoff")
plot_voronoi_cells(
    voronoi_data=individual,
    traj_data=traj,
    frame=frame,
    walkable_area=walkable_area,
    color_by_column=DENSITY_COL,
    axes=ax1,
    show_colorbar=False,
    vmin=0,
    vmax=10,
)

ax2 = fig.add_subplot(122, aspect="equal")
ax2.set_title("w cutoff")

plot_voronoi_cells(
    voronoi_data=individual_cutoff,
    traj_data=traj,
    frame=frame,
    walkable_area=walkable_area,
    color_by_column=DENSITY_COL,
    axes=ax2,
    show_colorbar=False,
    vmin=0,
    vmax=10,
)
cbar_ax = fig.add_axes([0.1, -0.05, 0.88, 0.05])

norm = mpl.colors.Normalize(vmin=0, vmax=10)
sm = plt.cm.ScalarMappable(cmap=plt.get_cmap("YlGn"), norm=norm)
sm.set_array([])
plt.colorbar(
    sm,
    cax=cbar_ax,
    shrink=0.1,
    label="$\\rho$ \\ 1/$m^2$",
    aspect=2,
    orientation="horizontal",
)

fig.tight_layout()
plt.show()

#### Compute actual Voronoi density

From these individual data we can now compute the Voronoi density $\rho_{voronoi}(t)$ in the known {class}`measurement area <geometry.MeasurementArea>` ($M$):

$$
    \rho_{voronoi}(t) = { \int\int \rho_{xy}(t) dxdy \over A(M)},
$$

where $\rho_{xy}(t) = 1 / A(V_i(t))$ is the individual density of each pedestrian, whose $V_i(t) \cap M$ and $A(M)$ the area of the {class}`measurement area <geometry.MeasurementArea>`.

##### Without cut-off

First, we compute the Voronoi density in the {class}`measurement area <geometry.MeasurementArea>` without a {class}`cut off <method_utils.CutOff>`:

In [ ]:
from pedpy import compute_voronoi_density

density_voronoi, intersecting = compute_voronoi_density(
    individual_voronoi_data=individual, measurement_area=measurement_area
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import PEDPY_ORANGE, plot_density

plot_density(density=density_voronoi, title="Voronoi density", color=PEDPY_ORANGE)
plt.show()

##### With cut-off

Second, we compute it now from the individual {class}`cut off <method_utils.CutOff>` Voronoi polygons:


In [ ]:
from pedpy import compute_voronoi_density

density_voronoi_cutoff, intersecting_cutoff = compute_voronoi_density(
    individual_voronoi_data=individual_cutoff, measurement_area=measurement_area
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import PEDPY_GREY, plot_density

plot_density(
    density=density_voronoi_cutoff,
    title="Voronoi density with cut-off",
    color=PEDPY_GREY,
)
plt.show()

### Comparison

Now we have obtained the mean density inside the {class}`measurement area <geometry.MeasurementArea>` with different methods.
To compare the results take a look at the following plot:

In [ ]:
import matplotlib.pyplot as plt

from pedpy import PEDPY_BLUE, PEDPY_GREY, PEDPY_ORANGE

fig = plt.figure()
plt.title("Comparison of different density methods")
plt.plot(
    classic_density.frame,
    classic_density.density,
    label="classic",
    color=PEDPY_BLUE,
)
plt.plot(
    density_voronoi.frame,
    density_voronoi.density,
    label="voronoi",
    color=PEDPY_ORANGE,
)
plt.plot(
    density_voronoi_cutoff.frame,
    density_voronoi_cutoff.density,
    label="voronoi with cutoff",
    color=PEDPY_GREY,
)
plt.xlabel("frame")
plt.ylabel("$\\rho$ / 1/$m^2$")
plt.grid()
plt.legend()
plt.show()

(passing_density)=
 ### Passing density (individual)

Another option to compute the individual density, is the passing density. 
For the computation it needs a {class}`measurement line <geometry.MeasurementLine>` and the distance to a second "virtual" {class}`measurement line <geometry.MeasurementLine>` which form a "virtual" {class}`measurement area <geometry.MeasurementArea>` ($M$).

```{eval-rst}
.. image:: /images/passing_area_from_lines.svg
    :width: 80 %
    :align: center
```

For each pedestrians now the frames when they enter and leave the virtual {class}`measurement area <geometry.MeasurementArea>` is computed. 
In this frame interval they have to be inside the {class}`measurement area <geometry.MeasurementArea>` continuously. 
They also need to enter and leave the {class}`measurement area <geometry.MeasurementArea>` via different {class}`measurement lines <geometry.MeasurementLine>`.
If leaving the area between the two lines, crossing the same line twice they will be ignored. 
For a better understanding, see the image below, where red parts of the trajectories are the detected ones inside the area. 
These frame intervals will be returned.

```{eval-rst}
.. image:: /images/frames_in_area.svg
    :width: 80 %
    :align: center
```

In this our example, we want to measure from the entrance of the bottleneck (top line) 1m towards the exit of the bottleneck (bottom line).
The set-up is shown below:

In [ ]:
from pedpy import compute_frame_range_in_area

frames_in_area, used_area = compute_frame_range_in_area(traj_data=traj, measurement_line=measurement_line, width=1.0)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_measurement_setup

plot_measurement_setup(
    measurement_areas=[used_area],
    measurement_lines=[
        measurement_line,
        MeasurementLine(shapely.offset_curve(measurement_line.line, 1.0)),
    ],
    ml_width=2,
    ma_line_width=0,
    ma_alpha=0.2,
    walkable_area=walkable_area,
).set_aspect("equal")
plt.show()

The passing density for each pedestrian, $\rho_{passing}(i)$, represents the average number of pedestrians present in the same {class}`measurement area <geometry.MeasurementArea>` $M$ during the same time interval ($[t_{in}(i), t_{out}(i)]$) as pedestrian $i$, divided by the area of that {class}`measurement area <geometry.MeasurementArea>` $A(M)$.

The computation is expressed as:

$$
    \rho_{passing}(i) = {1 \over {t_{out}(i) - t_{in}(i)}}
    \int_{t_{in}(i)}^{t_{out}(i)} \frac{N(t)}{A(M)} \, dt
$$

where:
- $t_{in}(i) = f_{in}(i) / fps$ is the time when pedestrian $i$ crosses the first line, 
- $t_{out}(i) = f_{out}(i) / fps$ is the time when they cross the second line,
- $f_{in}(i)$ and $f_{out}(i)$ are the respective frame indices for crossing the first and second lines, and 
- $fps$ is the frame rate of the {class}`trajectory data<trajectory_data.TrajectoryData>`.

To compute the passing density inside the bottleneck, the formula can be applied with the appropriate measurement area and time intervals:

In [ ]:
from pedpy import compute_frame_range_in_area, compute_passing_density

frames_in_area, used_area = compute_frame_range_in_area(traj_data=traj, measurement_line=measurement_line, width=1.0)
passing_density = compute_passing_density(density_per_frame=classic_density, frames=frames_in_area)

This provides a single density value for each pedestrian.
The following plot illustrates how individual densities are distributed within the bottleneck:

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_density_distribution

plot_density_distribution(density=passing_density, title="Individual density inside bottleneck")
plt.show()

### Line Density

Similar to the Voronoi density within a measurement area, the line density $\rho_{line}$ can be employed to determine the density at a measurement line $l$. In this approach, pedestrian densities are weighted based on the proportion of the total length of the line $w$ relative to the length of the intersection between the line and the Voronoi cell $w_i(t)$. 

The line density is mathematically defined as:

$$
\rho_\text{line}(t) = \sum_{{i \,|\, V_i(t) \cap l \neq \emptyset}} \frac{1}{A_i(t)} \cdot \frac{w_i(t)}{w},
$$

where:
- $A_i(t)$ is the area of the Voronoi cell $V_i(t)$ for pedestrian $i$ at time $t$, 
- $w_i(t)$ is the length of the intersection between the Voronoi cell $V_i(t)$ and the measurement line $l$, and
- $w$ is the total length of the measurement line $l$.

The line density can also be applied in scenarios where pedestrians approach the line from both sides. This allows for the analysis of data corresponding to different "species" of pedestrians (e.g., different directional flows or groups). However, in this example, only one species is considered, as the line is approached from a single side.

For further details, refer to [this](fundamental_diagram_at_measurement_line) Notebook.

In [ ]:
from pedpy import compute_line_density, compute_species

species = compute_species(
    trajectory_data=traj,
    individual_voronoi_polygons=individual_cutoff,
    frame_step=int(traj.frame_rate),
    measurement_line=measurement_line,
)

line_density = compute_line_density(
    individual_voronoi_polygons=individual_cutoff,
    measurement_line=measurement_line,
    species=species,
)

In [ ]:
from pedpy import plot_density

plot_density(density=line_density, title="density at measurement line")
plt.show()

## Speed

A further important measure in pedestrian dynamics is the speed of the pedestrians.
Low speeds can indicate congestions or other obstructions in the flow of the crowd.

### Individual speed

For computing the individuals speed at a specific frame $v_i(t)$, a specific frame step ($n$) is needed.
Together with the frame rate of the {class}`trajectory data<trajectory_data.TrajectoryData>` $fps$ the time frame $\Delta t$ for computing the speed becomes:

$$
    \Delta t = 2 n / fps
$$

This time step describes how many frames before and after the current position $X_{current}$ are used to compute the movement.
These positions are called $X_{future}$, $X_{past}$, respectively.

```{eval-rst}
.. image:: /images/speed_both.svg
    :width: 80 %
    :align: center
```

First computing the displacement between these positions $\bar{X}$.
This then can be used to compute the speed with:

\begin{align}
    \bar{X} &= X_{future} - X_{past} \\
    v_i(t) &= \frac{\bar{X}}{\Delta t}
\end{align}

When getting closer to the start, or end of the {class}`trajectory data<trajectory_data.TrajectoryData>`, it is not possible to use the full range of the frame interval for computing the speed.
For these cases *PedPy* offers three different methods to compute the speed:

1. exclude these parts
2. adaptively shrink the window in which the speed is computed
3. switch to one-sided window

#### Exclude border

When not enough frames available to compute the speed at the borders, for these parts no speed can be computed and they are ignored.


In [ ]:
from pedpy import SpeedCalculation, compute_individual_speed

frame_step = 25

individual_speed_exclude = compute_individual_speed(
    traj_data=traj,
    frame_step=frame_step,
    compute_velocity=True,
    speed_calculation=SpeedCalculation.BORDER_EXCLUDE,
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import PEDPY_GREEN

ped_id = 25

plt.figure()
plt.title(f"Speed time-series of a pedestrian {ped_id} (border excluded)")
single_individual_speed = individual_speed_exclude[individual_speed_exclude.id == ped_id]
plt.plot(
    single_individual_speed.frame,
    single_individual_speed.speed,
    color=PEDPY_GREEN,
)

plt.xlabel("frame")
plt.ylabel("v / m/s")
plt.show()

#### Adaptive border window

In the adaptive approach, it is checked how many frames $n$ are available to from $X_{current}$ to the end of the trajectory.
This number is then used on both sides to create a smaller symmetric window, which yields $X_{past}$ and $X_{future}$.
Now with the same principles as before the individual speed $v_i(t)$ can be computed.

```{eval-rst}
.. image:: /images/speed_border_adaptive_future.svg
    :width: 46 %
    
.. image:: /images/speed_border_adaptive_past.svg
    :width: 46 %
```

:::{important}
As the time interval gets smaller to the ends of the individual trajectories, the oscillations in the speed increase here.
:::

In [ ]:
from pedpy import SpeedCalculation, compute_individual_speed

individual_speed_adaptive = compute_individual_speed(
    traj_data=traj,
    frame_step=frame_step,
    compute_velocity=True,
    speed_calculation=SpeedCalculation.BORDER_ADAPTIVE,
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import PEDPY_RED

plt.figure()
plt.title(f"Speed time-series of an pedestrian {ped_id} (adaptive)")
single_individual_speed = individual_speed_adaptive[individual_speed_adaptive.id == ped_id]
plt.plot(
    single_individual_speed.frame,
    single_individual_speed.speed,
    color=PEDPY_RED,
)

plt.xlabel("frame")
plt.ylabel("v / m/s")
plt.show()

#### Single sided border window

In these cases, one of the end points to compute the movement becomes the current position $X_{current}$.
When getting too close to the start of the trajectory, the movement is computed from $X_{current}$ to $X_{future}$.
In the other case the movement is from $X_{past}$ to $X_{current}$.

$$
    v_i(t) = {|{X_{future} - X_{current}|}\over{ \frac{1}{2} \Delta t}} \text{, or } v_i(t) = {|{X_{current} - X_{past}|}\over{ \frac{1}{2} \Delta t}}
$$

```{eval-rst}
.. image:: /images/speed_border_single_sided_future.svg
    :width: 46 %
.. image:: /images/speed_border_single_sided_past.svg
    :width: 46 %
```

:::{important}
As at the edges of the trajectories the time interval gets halved, there may occur some jumps computed speeds at this point.
:::

In [ ]:
from pedpy import SpeedCalculation, compute_individual_speed

individual_speed_single_sided = compute_individual_speed(
    traj_data=traj,
    frame_step=frame_step,
    compute_velocity=True,
    speed_calculation=SpeedCalculation.BORDER_SINGLE_SIDED,
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import PEDPY_GREY

plt.figure()
plt.title(f"Speed time-series of an pedestrian {ped_id} (single sided)")
single_individual_speed = individual_speed_single_sided[individual_speed_single_sided.id == ped_id]
plt.plot(
    single_individual_speed.frame,
    single_individual_speed.speed,
    color=PEDPY_GREY,
)

plt.xlabel("frame")
plt.ylabel("v / m/s")
plt.show()

#### Comparison

To demonstrate the differences in the computed speeds, take a look at the following plot:


In [ ]:
import matplotlib.pyplot as plt

from pedpy import PEDPY_GREEN, PEDPY_GREY, PEDPY_RED

fig, ax = plt.subplots(1, 3, gridspec_kw={"width_ratios": [2, 1, 1]}, sharey=True, figsize=(12, 5))

fig.suptitle("Comparison of the different speed calculations at the borders")
speed_exclude = individual_speed_exclude[individual_speed_exclude.id == ped_id]
speed_adaptive = individual_speed_adaptive[individual_speed_adaptive.id == ped_id]
speed_single_sided = individual_speed_single_sided[individual_speed_single_sided.id == ped_id]

ax[0].plot(
    speed_single_sided.frame,
    speed_single_sided.speed,
    color=PEDPY_GREY,
    linewidth=3,
    label="single sided",
)
ax[0].plot(
    speed_adaptive.frame,
    speed_adaptive.speed,
    color=PEDPY_RED,
    linewidth=3,
    label="adaptive",
)
ax[0].plot(
    speed_exclude.frame,
    speed_exclude.speed,
    color=PEDPY_GREEN,
    linewidth=3,
    label="excluded",
)
ax[0].set_xlabel("frame")
ax[0].set_ylabel("v / m/s")
ax[0].legend()

ax[1].plot(
    speed_single_sided.frame[speed_single_sided.frame < speed_single_sided.frame.min() + 3 * frame_step],
    speed_single_sided.speed[speed_single_sided.frame < speed_single_sided.frame.min() + 3 * frame_step],
    color=PEDPY_GREY,
    linewidth=3,
)
ax[1].plot(
    speed_adaptive.frame[speed_adaptive.frame < speed_single_sided.frame.min() + 3 * frame_step],
    speed_adaptive.speed[speed_adaptive.frame < speed_single_sided.frame.min() + 3 * frame_step],
    color=PEDPY_RED,
    linewidth=3,
)
ax[1].plot(
    speed_exclude.frame[speed_exclude.frame < speed_single_sided.frame.min() + 3 * frame_step],
    speed_exclude.speed[speed_exclude.frame < speed_single_sided.frame.min() + 3 * frame_step],
    color=PEDPY_GREEN,
    linewidth=3,
)
ax[1].set_xlabel("frame")

ax[2].plot(
    speed_single_sided.frame[speed_single_sided.frame > speed_single_sided.frame.max() - 3 * frame_step],
    speed_single_sided.speed[speed_single_sided.frame > speed_single_sided.frame.max() - 3 * frame_step],
    color=PEDPY_GREY,
    linewidth=3,
)
ax[2].plot(
    speed_adaptive.frame[speed_adaptive.frame > speed_single_sided.frame.max() - 3 * frame_step],
    speed_adaptive.speed[speed_adaptive.frame > speed_single_sided.frame.max() - 3 * frame_step],
    color=PEDPY_RED,
    linewidth=3,
)
ax[2].plot(
    speed_exclude.frame[speed_exclude.frame > speed_single_sided.frame.max() - 3 * frame_step],
    speed_exclude.speed[speed_exclude.frame > speed_single_sided.frame.max() - 3 * frame_step],
    color=PEDPY_GREEN,
    linewidth=3,
)

ax[2].set_xlabel("frame")
plt.show()

#### Individual speed in specific movement direction

It is also possible to compute the individual speed in a specific direction $d$, for this the movement $\bar{X}$ is projected onto the desired movement direction. $\bar{X}$ and $\Delta t$ are computed as described above. 
Hence, the speed then becomes:

$$
    v_i(t) = {{|\boldsymbol{proj}_d\; \bar{X}|} \over {\Delta t}}
$$

```{eval-rst}
.. image:: /images/speed_movement_direction.svg
    :width: 80 %
    :align: center
```

:::{important}
When using a specific direction, the computed speed may become negative.
:::

In [ ]:
individual_speed_direction = compute_individual_speed(
    traj_data=traj,
    frame_step=5,
    movement_direction=np.array([0, -1]),
    compute_velocity=True,
    speed_calculation=SpeedCalculation.BORDER_SINGLE_SIDED,
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import PEDPY_BLUE, PEDPY_GREEN, PEDPY_GREY, PEDPY_RED

colors = [PEDPY_BLUE, PEDPY_GREY, PEDPY_RED, PEDPY_GREEN]
ped_ids = [10, 20, 17, 70]

fig = plt.figure()
plt.title("Velocity time-series of an excerpt of the pedestrians in a specific direction")
for color, ped_id in zip(colors, ped_ids, strict=False):
    single_individual_speed = individual_speed_direction[individual_speed_direction.id == ped_id]
    plt.plot(
        single_individual_speed.frame,
        single_individual_speed.speed,
        color=color,
    )

plt.xlabel("frame")
plt.ylabel("v / m/s")
plt.show()

(mean_speed)=
 ### Mean speed

Now, that we have computed the individual's speed, we want to compute the mean speed in the already used {class}`measurement area <geometry.MeasurementArea>` $M$ closely in front of the bottleneck.
The mean speed is defined as

$$
    v_{mean}(t) = {{1} \over {N}} \sum_{i \in P_M} v_i(t), 
$$

where $P_M$ are all pedestrians inside the {class}`measurement area <geometry.MeasurementArea>`, and $N$ the number of pedestrians inside the {class}`measurement area <geometry.MeasurementArea>` ($|P_M|$).

:::{important}
The mean speed can only be computed when for each pedestrian inside the {class}`measurement area <geometry.MeasurementArea>` also a speed $v_i(t)$ is computed, when using the exclude or adaptive approach this might not be the case.
Then some extra processing steps are needed, to avoid this use the single sided approach. 
:::

This can be as follows with *PedPy*:
 

In [ ]:
from pedpy import compute_mean_speed_per_frame

mean_speed = compute_mean_speed_per_frame(
    traj_data=traj,
    measurement_area=measurement_area,
    individual_speed=individual_speed_single_sided,
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import PEDPY_BLUE, plot_speed

plot_speed(
    speed=mean_speed.speed,
    title="Mean speed in front of the bottleneck",
    color=PEDPY_BLUE,
)
plt.show()

The same can be now computed, using the speed in a movement direction as basis:

In [ ]:
mean_speed_direction = compute_mean_speed_per_frame(
    traj_data=traj,
    measurement_area=measurement_area,
    individual_speed=individual_speed_direction,
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import PEDPY_RED, plot_speed

plot_speed(
    speed=mean_speed_direction.speed,
    title="Mean speed in specific direction in front of the bottleneck",
    color=PEDPY_RED,
)
plt.show()

(voronoi_speed)=
 ### Voronoi speed

A further approach to compute average speed $v_{voronoi}(t)$ in an area by weighting the individuals speed by the size of their corresponding Voronoi polygon $V_i$ inside the {class}`measurement area <geometry.MeasurementArea>` $M$.
The individuals speed are weighted by the proportion of their Voronoi cell $V_i$ and the intersection with the {class}`measurement area <geometry.MeasurementArea>` $V_i \cap M$.

The Voronoi speed $v_{voronoi}(t)$ is defined as

$$
        v_{voronoi}(t) = { \int\int v_{xy}(t) dxdy \over A(M)},
$$

where $v_{xy}(t) = v_i(t)$ is the individual speed of each pedestrian, whose $V_i(t) \cap M$ and $A(M)$ the area of the {class}`measurement area <geometry.MeasurementArea>`.

```{eval-rst}
.. image:: /images/voronoi_density.svg
    :width: 60 %
    :align: center
```

:::{important}
The Voronoi speed can only be computed when for each pedestrian inside the {class}`measurement area <geometry.MeasurementArea>` also a speed $v_i(t)$ is computed, when using the exclude or adaptive approach this might not be the case.
Then some extra processing steps are needed, to avoid this use the single sided approach. 
:::

This can be done in *PedPy* with:

In [ ]:
from pedpy import compute_voronoi_speed

voronoi_speed = compute_voronoi_speed(
    traj_data=traj,
    individual_voronoi_intersection=intersecting,
    individual_speed=individual_speed_single_sided,
    measurement_area=measurement_area,
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import PEDPY_ORANGE, plot_speed

plot_speed(
    speed=voronoi_speed.speed,
    title="Voronoi speed in front of the bottleneck",
    color=PEDPY_ORANGE,
)
plt.show()

Analogously, this can be done with the speed in a specific direction with:

In [ ]:
voronoi_speed_direction = compute_voronoi_speed(
    traj_data=traj,
    individual_voronoi_intersection=intersecting,
    individual_speed=individual_speed_direction,
    measurement_area=measurement_area,
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import PEDPY_GREY, plot_speed

plot_speed(
    speed=voronoi_speed.speed,
    title="Voronoi velocity in specific direction in front of the bottleneck",
    color=PEDPY_GREY,
)
plt.show()

### Comparison mean speed vs Voronoi speed

We now computed the speed with different methods, this plot shows what the different results look like compared to each other:

In [ ]:
import matplotlib.pyplot as plt

from pedpy import PEDPY_BLUE, PEDPY_GREY, PEDPY_ORANGE, PEDPY_RED

plt.figure(figsize=(8, 6))
plt.title("Comparison of different speed methods")
plt.plot(
    voronoi_speed.frame,
    voronoi_speed.speed,
    label="Voronoi",
    color=PEDPY_ORANGE,
)
plt.plot(
    voronoi_speed_direction.frame,
    voronoi_speed_direction.speed,
    label="Voronoi direction",
    color=PEDPY_GREY,
)
plt.plot(
    mean_speed.frame,
    mean_speed.speed,
    label="classic",
    color=PEDPY_BLUE,
)
plt.plot(
    mean_speed_direction.frame,
    mean_speed_direction.speed,
    label="classic direction",
    color=PEDPY_RED,
)
plt.xlabel("frame")
plt.ylabel("v / m/s")
plt.legend()
plt.grid()
plt.show()

### Line speed

Similar to the Voronoi speed in a measuring area, the line speed can be used to determine the speed $v_{line}$ at a measurement line $l$. The speeds of individuals are weighted based on the proportion of the total length of the line $w$ to the length of the intersection between the line and the Voronoi cell $w_i(t)$.
The speed at the line is defined as:
$$
v_\text{line}(t) = \sum_{i | A_i(t) \cap l \neq \emptyset} v_i(t) \cdot n_l \cdot \frac{w_i(t)}{w}
$$

The line speed only includes the speed orthogonal to the line.
Therefore, the speed of the pedestrians is multiplied by the normal vector of the measuring line $n_l$.

The line speed can also be used when pedestrians are approaching the line from both sides. In this case it is possible to analyze the data of both "species". In this example, there will be only one species as the line is only approached from one side.  

For further details check out [this](fundamental_diagram_at_measurement_line) Notebook.

In [ ]:
from pedpy import compute_line_speed

line_speed = compute_line_speed(
    individual_speed=individual_speed_single_sided,
    species=species,
    measurement_line=measurement_line,
    individual_voronoi_polygons=individual_cutoff,
)

In [ ]:
from pedpy import plot_speed
from pedpy.column_identifier import SPEED_COL

plot_speed(speed=line_speed[SPEED_COL])
plt.show()

### Passing speed (individual)

With the same principles as described in [passing density](passing_density), the individual speeds $v^i_{passing}$ is defined as

$$
    v^i_{passing} = \frac{d}{t_{out}-t_{in}},
$$

where $d$ is the distance between the two {class}`measurement lines <geometry.MeasurementLine>`.

In *PedPy* this can be done with: 

In [ ]:
from pedpy import compute_frame_range_in_area, compute_passing_speed

passing_offset = 1.0
frames_in_area, _ = compute_frame_range_in_area(traj_data=traj, measurement_line=measurement_line, width=passing_offset)
passing_speed = compute_passing_speed(
    frames_in_area=frames_in_area,
    frame_rate=traj.frame_rate,
    distance=passing_offset,
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_speed_distribution

plot_speed_distribution(speed=passing_speed, title="Individual speed in bottleneck")
plt.show()

## Flow

Another important metric, when analyzing pedestrian flows is the flow itself.
It describes how many persons cross a line in a given time.
From this potential bottlenecks or congestion can be derived.

### N-t diagram at bottleneck

To get a first impression of the flow at the bottleneck we look at the N-t diagram, which shows how many pedestrian have crossed the {class}`measurement line <geometry.MeasurementLine>` at a specific time.

In [ ]:
from pedpy import compute_n_t

nt, crossing = compute_n_t(
    traj_data=traj,
    measurement_line=measurement_line,
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_nt

plot_nt(nt=nt, title="N-t at bottleneck")
plt.show()

### Flow at bottleneck

From the N-t data we then can compute the flow at the bottleneck.

For the computation of the flow we look at frame intervals $\Delta frame$ in which the flow is computed.
The first intervals starts, when the first person crossed the {class}`measurement line <geometry.MeasurementLine>`.
The next interval always starts at the time when the last person in the previous frame interval crossed the line.

```{eval-rst}
.. image:: /images/flow.svg
    :align: center
    :width: 80 %
```

In each of the time interval it is checked, if any person has crossed the line, if yes, a flow $J$ can be computed.
From the first frame the line was crossed $f^{\Delta frame}_1$, the last frame someone crossed the line $f^{\Delta frame}_N$ the length of the frame interval $\Delta f$ can be computed:

$$
    \Delta f = f^{\Delta frame}_N - f^{\Delta frame}_1
$$

This directly together with the frame rate of the trajectory $fps$ gives the time interval $\Delta t$:

$$
    \Delta t = \Delta f / fps
$$

Given the number of pedestrian crossing the line is given by $N^{\Delta frame}$, the flow $J$ becomes:

$$
    J = \frac{N^{\Delta frame}}{\Delta t}
$$

```{eval-rst}
.. image:: /images/flow_zoom.svg
    :align: center
    :width: 60 %
```

At the same time also the mean speed of the pedestrian when crossing the line is given by:

$$
    v_{crossing} = {1 \over N^{\Delta t} } \sum^{N^{\Delta t}}_{i=1} v_i(t)
$$

To compute the flow and mean speed when passing the line with *PedPy* use:

In [ ]:
from pedpy import compute_flow

delta_frame = 100
flow = compute_flow(
    nt=nt,
    crossing_frames=crossing,
    individual_speed=individual_speed_single_sided,
    delta_frame=delta_frame,
    frame_rate=traj.frame_rate,
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_flow

plot_flow(
    flow=flow,
    title="Crossing velocities at the corresponding flow at bottleneck",
)
plt.show()

## Acceleration



### Individual acceleration

Compute the individual acceleration for each pedestrian.

For computing the individuals' acceleration at a specific frame $a_i(t_k)$,
a specific frame step ($n$) is needed.
Together with the {class}`trajectory data<trajectory_data.TrajectoryData>` $fps$ of
the trajectory data $fps$ the time frame $\Delta t$ for
computing the speed becomes:

$$
\Delta t = 2 n / fps
$$

This time step describes how many frames before and after the current
position $X(t_k)$ are used to compute the movement.
These positions are called $X(t_{k+n})$, $X(t_{k-n})$
respectively.

In order to compute the acceleration at time $t_k$, we first calculate the
displacements $\bar{X}$ around $t_{k+n}$ and $t_{k-n}$:

$$
\bar{X}(t_{k+n}) = X(t_{k+2n}) - X(t_{k})
$$
$$
\bar{X}(t_{k-n}) = X(t_{k}) - X(t_{k-2n})
$$

The acceleration is then calculated from the difference of the displacements

$$
\Delta\bar{X}(t_k) = \bar{X}(t_{k+n}) - \bar{X}(t_{k-n})
$$

divided by the square of the time interval $\Delta t$:

$$
a_i(t_k) = \Delta\bar{X}(t_k) / \Delta t^{2}
$$

When getting closer to the start, or end of the trajectory data, it is not
possible to use the full range of the frame interval for computing the
acceleration. For these cases *PedPy* offers a method to compute
the acceleration:

**Exclude border:**

When not enough frames available to compute the speed at the borders, for
these parts no acceleration can be computed and they are ignored. Use
`acceleration_calculation=AccelerationCalculation.BORDER_EXCLUDE`.



In [ ]:
from pedpy import AccelerationCalculation, compute_individual_acceleration

frame_step = 25

individual_acceleration_exclude = compute_individual_acceleration(
    traj_data=traj,
    frame_step=frame_step,
    compute_acceleration_components=True,
    acceleration_calculation=AccelerationCalculation.BORDER_EXCLUDE,
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import PEDPY_GREEN

ped_id = 50

plt.figure()
plt.title(f"Acceleration time-series of a pedestrian {ped_id} (border excluded)")
single_individual_acceleration = individual_acceleration_exclude[individual_acceleration_exclude.id == ped_id]
plt.plot(
    single_individual_acceleration.frame,
    single_individual_acceleration.acceleration,
    color=PEDPY_GREEN,
)

plt.xlabel("frame")
plt.ylabel("a / $m/s^2$")
plt.show()

### Individual acceleration in specific movement direction:

It is also possible to compute the individual acceleration in a specific direction
$d$, for this the movement $\Delta\bar{X}$ is projected onto the
desired movement direction. $\Delta\bar{X}$ and $\Delta t$ are
computed as described above. Hence, the acceleration then becomes:

$$
a_i(t) = {{|\boldsymbol{proj}_d\; \Delta\bar{X}|} \over {\Delta t^{2}}}
$$

If `compute_acceleration_components` is `True` also $\Delta\bar{X}$ is returned.

In [ ]:
individual_acceleration_direction = compute_individual_acceleration(
    traj_data=traj,
    frame_step=5,
    movement_direction=np.array([0, -1]),
    compute_acceleration_components=True,
    acceleration_calculation=AccelerationCalculation.BORDER_EXCLUDE,
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import PEDPY_BLUE, PEDPY_GREEN, PEDPY_GREY, PEDPY_RED

colors = [PEDPY_BLUE, PEDPY_GREY, PEDPY_RED, PEDPY_GREEN]
ped_ids = [10, 20, 17, 70]

fig = plt.figure()
plt.title("Acceleration time-series of an excerpt of the pedestrians in a specific direction")
for color, ped_id in zip(colors, ped_ids, strict=False):
    single_individual_acceleration = individual_acceleration_direction[individual_acceleration_direction.id == ped_id]
    plt.plot(
        single_individual_acceleration.frame,
        single_individual_acceleration.acceleration,
        color=color,
    )

plt.xlabel("frame")
plt.ylabel("a / $m/s^2$")
plt.show()

### Mean acceleration

Compute mean acceleration per frame inside a given measurement area.

Computes the mean acceleration $a_{mean}(t)$ inside the measurement area from
the given individual acceleration data $a_i(t)$ (see
{func}`~acceleration_calculator.compute_individual_acceleration` for
details of the computation). The mean acceleration $a_{mean}$ is defined as

$$
a_{mean}(t) = {{1} \over {N}} \sum_{i \in P_M} a_i(t),
$$

where $P_M$ are all pedestrians inside the measurement area, and
$N$ the number of pedestrians inside the measurement area (
$|P_M|$).


:::{important}
The mean acceleration can only be computed when for each pedestrian inside the {class}`measurement area <geometry.MeasurementArea>` also an acceleration $a_i(t)$ is computed, when using the exclude or adaptive approach this might not be the case.
Therefore, further processing steps are needed to ensure that the trajectory data and the individual acceleration data overlap, e.g. as in the example below:
:::

In [ ]:
from pedpy import TrajectoryData

traj_idx = pd.MultiIndex.from_frame(traj.data[["id", "frame"]])
acc_idx = pd.MultiIndex.from_frame(individual_acceleration_exclude[["id", "frame"]])
# get intersecting rows in id and frame
common_idx = traj_idx.intersection(acc_idx)
traj_common = traj.data.set_index(["id", "frame"]).reindex(common_idx).reset_index()

fr = traj.frame_rate
traj_exclude = TrajectoryData(data=traj_common, frame_rate=fr)

Now the mean acceleration can be computed using *PedPy* with:

In [ ]:
from pedpy import compute_mean_acceleration_per_frame

mean_acceleration = compute_mean_acceleration_per_frame(
    traj_data=traj_exclude,
    measurement_area=measurement_area,
    individual_acceleration=individual_acceleration_exclude,
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import PEDPY_BLUE, plot_acceleration

plot_acceleration(
    acceleration=mean_acceleration.acceleration,
    title="Mean acceleration in front of the bottleneck",
    color=PEDPY_BLUE,
)
plt.show()

The same can be now computed, using the speed in a movement direction as basis:

In [ ]:
mean_acceleration_direction = compute_mean_acceleration_per_frame(
    traj_data=traj_exclude,
    measurement_area=measurement_area,
    individual_acceleration=individual_acceleration_direction,
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import PEDPY_RED, plot_acceleration

plot_acceleration(
    acceleration=mean_acceleration_direction.acceleration,
    title="Mean acceleration in specific direction in front of the bottleneck",
    color=PEDPY_RED,
)
plt.show()

### Voronoi acceleration
Compute the Voronoi acceleration per frame inside the measurement area.

Computes the Voronoi acceleration $a_{voronoi}(t)$ inside the measurement
area $M$ from the given individual acceleration data $a_i(t)$ (see
{func}`~acceleration_calculator.compute_individual_acceleration` for
details of the computation) and their individual Voronoi intersection data
(from {func}`~density_calculator.compute_voronoi_density`).
The individuals' accelerations are weighted by the proportion of their Voronoi cell
$V_i$ and the intersection with the measurement area
$V_i \cap M$.

The Voronoi acceleration $a_{voronoi}(t)$ is defined as

$$
a_{voronoi}(t) = { \int\int a_{xy}(t) dxdy \over A(M)},
$$

where $a_{xy}(t) = a_i(t)$ is the individual acceleration of
each pedestrian, whose $V_i(t) \cap M$ and $A(M)$ the area of
the measurement area.

This can be as follows with *PedPy*:

:::{important}
The Voronoi acceleration can only be computed when for each pedestrian inside the {class}`measurement area <geometry.MeasurementArea>` also an acceleration $a_i(t)$ is computed, when using the exclude or adaptive approach this might not be the case.

Therefore, further processing steps are needed to ensure that the trajectory data, the individual acceleration dara as well as the intersecting Voronoi cell data overlap. We assume that the trajectory data is already overlapping (see example above). The intersecting voronoi cells can be filtered, e.g., as in the example below:
:::

In [ ]:
intersecting_idx = pd.MultiIndex.from_frame(intersecting[["id", "frame"]])
acc_idx = pd.MultiIndex.from_frame(individual_acceleration_exclude[["id", "frame"]])
# get intersecting rows in id and frame
common_idx = intersecting_idx.intersection(acc_idx)
intersecting_exclude = intersecting.set_index(["id", "frame"]).reindex(common_idx).reset_index()

Now the Voronoi acceleration can be computed using *PedPy* with:

In [ ]:
from pedpy import compute_voronoi_acceleration

voronoi_acceleration = compute_voronoi_acceleration(
    traj_data=traj_exclude,
    individual_voronoi_intersection=intersecting_exclude,
    individual_acceleration=individual_acceleration_exclude,
    measurement_area=measurement_area,
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import PEDPY_ORANGE, plot_acceleration

plot_acceleration(
    acceleration=voronoi_acceleration.acceleration,
    title="Voronoi acceleration in front of the bottleneck",
    color=PEDPY_ORANGE,
)
plt.show()

Analogously, this can be done with the acceleration in a specific direction with:

In [ ]:
voronoi_acceleration_direction = compute_voronoi_acceleration(
    traj_data=traj_exclude,
    individual_voronoi_intersection=intersecting_exclude,
    individual_acceleration=individual_acceleration_direction,
    measurement_area=measurement_area,
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import PEDPY_GREY, plot_acceleration

plot_acceleration(
    acceleration=voronoi_acceleration.acceleration,
    title="Voronoi acceleration in specific direction in front of the bottleneck",
    color=PEDPY_GREY,
)
plt.show()

### Comparison mean acceleration vs Voronoi acceleration

We now computed the acceleration with different methods, this plot shows what the different results look like compared to each other:

In [ ]:
import matplotlib.pyplot as plt

from pedpy import PEDPY_BLUE, PEDPY_GREY, PEDPY_ORANGE, PEDPY_RED

plt.figure(figsize=(8, 6))
plt.title("Comparison of different acceleration methods")
plt.plot(
    voronoi_acceleration.frame,
    voronoi_acceleration.acceleration,
    label="Voronoi",
    color=PEDPY_ORANGE,
)
plt.plot(
    voronoi_acceleration_direction.frame,
    voronoi_acceleration_direction.acceleration,
    label="Voronoi direction",
    color=PEDPY_GREY,
)
plt.plot(
    mean_acceleration.frame,
    mean_acceleration.acceleration,
    label="classic",
    color=PEDPY_BLUE,
)
plt.plot(
    mean_acceleration_direction.frame,
    mean_acceleration_direction.acceleration,
    label="classic direction",
    color=PEDPY_RED,
)
plt.xlabel("frame")
plt.ylabel("a / $m/s^2$")
plt.legend()
plt.grid()
plt.show()

### Line flow

The line flow is derived from the line speed and line density. It can is used to determine the flow at a measurement line $l$. The values of individuals are weighted based on the proportion of the total length of the line $w$ to the length of the intersection between the line and the Voronoi cell $w_i(t)$.
The flow at the line is defined as:

$$
j_\text{line}(t) = \sum_{{i | A_i(t) \cap l \neq \emptyset}} v_i(t) \cdot n_l \cdot \frac{1}{A_i(t)} \cdot \frac{w_i(t)}{w}
$$

The line flow only includes the speed orthogonal to the line. Therefore, the speed of the pedestrians $v_i(t)$ is multiplied by the normal vector of the measuring line $n_l$.

The line flow can also be used when pedestrians are approaching the line from both sides.In this case it is possible to analyze the data of both "species". In this example there will be only one species as the line is only approached from one side. 

For further details check out [this](fundamental_diagram_at_measurement_line) Notebook.

In [ ]:
from pedpy import compute_line_flow

line_flow = compute_line_flow(
    individual_voronoi_polygons=individual_cutoff,
    species=species,
    measurement_line=measurement_line,
    individual_speed=individual_speed_single_sided,
)

In [ ]:
from pedpy.column_identifier import FLOW_COL

plt.plot(line_flow[FRAME_COL], line_flow[FLOW_COL])
plt.title("flow on line")
plt.xlabel("frame")
plt.ylabel("$J$")
plt.show()

## Neighborhood

To analyze which pedestrians are close to each other, it is possible to compute the neighbors of each pedestrian.
We define two pedestrians as neighbors if their Voronoi polygons ($V_i$, $V_j$) touch at some point, in case of *PedPy* they are touching if their distance is below 1mm.
As basis for the computation one can either use the uncut or cut Voronoi polygons.
When using the uncut Voronoi polygons, pedestrian may be detected as neighbors even when their distance is quite large in low density situation.
Therefore, it is recommended to use the cut Voronoi polygons, where the cut-off radius can be used to define a maximal distance between neighboring pedestrians.

### Neighborhood computation

To compute the neighbors in *PedPy* use:

In [ ]:
from pedpy import compute_neighbors

neighbors = compute_neighbors(individual_cutoff, as_list=False)
neighbors[0:5]

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_neighborhood

plot_neighborhood(
    pedestrian_id=8,
    voronoi_data=individual_cutoff,
    frame=350,
    neighbors=neighbors,
    walkable_area=walkable_area,
).set_aspect("equal")
plt.show()

:::{important}
For legacy reasons the function {func}`compute_neighbors <method_utils.compute_neighbors>` works also without specifing `as_list` (defaults to `True`). 
We highly discourage using this, as its result is harder to be used in further computations.
Use 'as_list=False' instead.
The default value may change in future versions of *PedPy*.
:::

In [ ]:
neighbors_as_list = compute_neighbors(individual_cutoff)
neighbors_as_list[0:5]

### Distance to neighbors

For computing the distance between neighbors, *PedPy* offers a dedicated function:

In [ ]:
from pedpy import compute_neighbor_distance

neighbor_distance = compute_neighbor_distance(traj_data=traj, neighborhood=neighbors)
neighbor_distance[0:5]

:::{note}
The resulting {class}`~pandas.DataFrame` is symmetric. 
If pedestrian A is a neighbor of pedestrian B, then pedestrian B is also a neighbor of pedestrian A.
Consequently, the distance between both appears twice in the DataFrame.
:::

## Distance to entrance/Time to entrance

An indicator to detect congestions or jams in {class}`trajectory data<trajectory_data.TrajectoryData>` are distance/time to crossing.
It shows how much time until the crossing of the {class}`measurement line <geometry.MeasurementLine>` is left and how big the distance to that line is.

In *PedPy* this can be done with:

In [ ]:
from pedpy import compute_time_distance_line

df_time_distance = compute_time_distance_line(traj_data=traj, measurement_line=measurement_line)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_time_distance

plot_time_distance(
    time_distance=df_time_distance,
    title="Distance to entrance/Time to entrance",
    frame_rate=traj.frame_rate,
)
plt.show()

You can also color the lines based on speed values. 
To achieve this, simply provide a speed DataFrame as an argument to the function:

In [ ]:
speed = compute_individual_speed(traj_data=traj, frame_step=5)
plot_time_distance(
    time_distance=df_time_distance,
    title="Distance to entrance/Time to entrance - colored",
    frame_rate=traj.frame_rate,
    speed=speed,
)
plt.show()

## Profiles

For the computation of the profiles the given {class}`walkable area <geometry.WalkableArea>` or {class}`axis-aligned measurement area <geometry.AxisAlignedMeasurementArea>` is divided into square grid cells.
Each of these grid cells is then used as a {class}`measurement area <geometry.MeasurementArea>`  to compute the density and speed.

When using the whole walkable area for the computing the profiles, the used grid will look like this:
```{eval-rst}
.. image:: /images/profile_grid.svg
    :width: 70 %
    :align: center
```

:::{note}
As this is a quite compute heavy operation, it is suggested to reduce the trajectories to the important areas and limit the input data to the most relevant frame interval. 

In *PedPy* it's possible to pass an axis-aligned measurement area as option to the profile computation, this will retrict the computation to this area.
:::


As the profiles are computed on a square axis-aligned grid, the computation can only be restriced to axis-aligned measurement areas! 
For an arbritraty shaped measurement area, internally the grid would be created for the bounding box of that measurement area. 
We decided to make this step explicit and visible to the users.
The resulting grid of such an axis-aligned measurement area, could look like this:

```{eval-rst}
.. image:: /images/profile_axis_aligned_measurement_area_with_grid.svg
    :width: 70 %
    :align: center
```


In [ ]:
from pedpy import (
    Cutoff,
    SpeedCalculation,
    compute_individual_speed,
    compute_individual_voronoi_polygons,
)

individual_cutoff = compute_individual_voronoi_polygons(
    traj_data=traj,
    walkable_area=walkable_area,
    cut_off=Cutoff(radius=0.8, quad_segments=3),
)

individual_speed = compute_individual_speed(
    traj_data=traj,
    frame_step=5,
    speed_calculation=SpeedCalculation.BORDER_SINGLE_SIDED,
)

profile_data = individual_speed.merge(individual_cutoff, on=[ID_COL, FRAME_COL])
profile_data = profile_data.merge(traj.data, on=[ID_COL, FRAME_COL])

In [ ]:
from pedpy import (
    compute_grid_cell_polygon_intersection_area,
    get_grid_cells,
)

grid_size = 0.4
grid_cells, _, _ = get_grid_cells(walkable_area=walkable_area, grid_size=grid_size)

min_frame_profiles = 250  # We use here just an excerpt of the
max_frame_profiles = 400  # trajectory data to reduce compute time

profile_data = profile_data[profile_data.frame.between(min_frame_profiles, max_frame_profiles)]

# Compute the grid intersection area for the resorted profile data (they have the same sorting)
# for usage in multiple calls to not run the compute heavy operation multiple times
(
    grid_cell_intersection_area,
    resorted_profile_data,
) = compute_grid_cell_polygon_intersection_area(data=profile_data, grid_cells=grid_cells)

Create an axis-aligned measurement area to speed-up the profile computation. This can either be done directly from the bounding box of the desired measurement area:

In [ ]:
from pedpy import AxisAlignedMeasurementArea

profile_measurement_area = AxisAlignedMeasurementArea(-1.5, 0.5, 1.5, 2.8)

Alternatively, an axis-aligned measurement can be created from a given measurement area:

In [ ]:
from pedpy import AxisAlignedMeasurementArea, MeasurementArea

measurement_area = MeasurementArea([(-1.5, 0.5), (1.5, 0.5), (1.5, 2.8), (-1.5, 2.8)])
profile_measurement_area = AxisAlignedMeasurementArea.from_measurement_area(measurement_area=measurement_area)

In [ ]:
from pedpy import plot_measurement_setup

plot_measurement_setup(
    walkable_area=walkable_area,
    measurement_areas=[profile_measurement_area],
    ma_line_width=2,
    ma_alpha=0.2,
).set_aspect("equal")
plt.show()

In [ ]:
from pedpy import (
    compute_grid_cell_polygon_intersection_area,
    get_grid_cells,
)

grid_cells_measurement_area, _, _ = get_grid_cells(
    axis_aligned_measurement_area=profile_measurement_area,
    grid_size=grid_size,
)

min_frame_profiles = 250  # We use here just an excerpt of the
max_frame_profiles = 400  # trajectory data to reduce compute time

profile_data = profile_data[profile_data.frame.between(min_frame_profiles, max_frame_profiles)]

# Compute the grid intersection area for the resorted profile data (they have the same sorting)
# for usage in multiple calls to not run the compute heavy operation multiple times
(
    grid_cell_intersection_area_measurement_area,
    resorted_profile_data_measurement_area,
) = compute_grid_cell_polygon_intersection_area(data=profile_data, grid_cells=grid_cells_measurement_area)

### Speed Profiles

This documentation describes the methods available for computing speed profiles within a specified area, focusing on pedestrian movements. Four distinct methods are detailed: Voronoi, Arithmetic, Mean, and Gauss speed profiles.

**Voronoi speed profile**

The Voronoi speed profile $v_{\text{voronoi}}$ is computed from a weighted mean of pedestrian speeds.
 It utilizes the area overlap between a pedestrian's Voronoi cell ($V_i$) and the grid cell ($c$). The weight corresponds to the fraction of the Voronoi cell residing within the grid cell, thereby integrating speed across this intersection:
 
$$
        v_{\text{voronoi}} = { \int\int v_{xy} dxdy \over A(c)},
$$

where $A(c)$ represents the area of the grid cell $c$.

**Arithmetic Voronoi speed profile**

The arithmetic Voronoi speed $v_{\text{arithmetic}}$ is computed as the mean of each pedestrian's speed ($v_i$), whose Voronoi cell $V_i$ intersects with grid cell $c$:

$$
        v_{\text{arithmetic}} = \frac{1}{N} \sum_{i \in V_i \cap P_c} v_i,
$$

with $N$ being the total number of pedestrians whose Voronoi cells overlap with grid cell $c$.

**Mean speed profile**

The mean speed profile is computed by the average speed of all pedestrians $P_c$ present within a grid cell $c$:

$$
        v_{\text{mean}} = \frac{1}{N} \sum_{i \in P_c} v_i
$$

where $N$ denotes the number of pedestrians in grid cell $c$.

**Gauss speed profiles**

Calculates a speed profile based on Gaussian weights for an array of pedestrian locations and velocities. The speed, weighted at a grid cell $c$ considering its distance $\delta = \boldsymbol{r}_i - \boldsymbol{c}$ from an agent, is determined as follows:
    
$$     
        v_{\text{gauss}} = \frac{\sum_{i=1}^{N}{\big(w_i\cdot v_i\big)}}{\sum_{i=1}^{N} w_i},
$$
with 
$w_i = \frac{1} {\sigma \cdot \sqrt{2\pi}} \exp\big(-\frac{\delta^2}{2\sigma^2}\big)\; \text{and}\; \sigma = \frac{FWHM}{2\sqrt{2\ln(2)}}.
$

In [ ]:
from pedpy import SpeedMethod, compute_speed_profile

voronoi_speed_profile = compute_speed_profile(
    data=resorted_profile_data,
    walkable_area=walkable_area,
    grid_intersections_area=grid_cell_intersection_area,
    grid_size=grid_size,
    speed_method=SpeedMethod.VORONOI,
)

arithmetic_speed_profile = compute_speed_profile(
    data=resorted_profile_data,
    walkable_area=walkable_area,
    grid_intersections_area=grid_cell_intersection_area,
    grid_size=grid_size,
    speed_method=SpeedMethod.ARITHMETIC,
)

mean_speed_profile = compute_speed_profile(
    data=profile_data,
    walkable_area=walkable_area,
    grid_size=grid_size,
    speed_method=SpeedMethod.MEAN,
)

gauss_speed_profile = compute_speed_profile(
    data=profile_data,
    walkable_area=walkable_area,
    grid_size=grid_size,
    gaussian_width=0.5,
    speed_method=SpeedMethod.GAUSSIAN,
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_profiles

fig, ((ax0, ax1), (ax2, ax3)) = plt.subplots(nrows=2, ncols=2)
fig.suptitle("Speed profile")
cm = plot_profiles(
    walkable_area=walkable_area,
    profiles=voronoi_speed_profile,
    axes=ax0,
    label="v / m/s",
    vmin=0,
    vmax=1.5,
    title="Voronoi",
)
cm = plot_profiles(
    walkable_area=walkable_area,
    profiles=arithmetic_speed_profile,
    axes=ax1,
    label="v / m/s",
    vmin=0,
    vmax=1.5,
    title="Arithmetic",
)
cm = plot_profiles(
    walkable_area=walkable_area,
    profiles=gauss_speed_profile,
    axes=ax2,
    label="v / m/s",
    vmin=0,
    vmax=1.5,
    title="Gauss",
)
cm = plot_profiles(
    walkable_area=walkable_area,
    profiles=mean_speed_profile,
    axes=ax3,
    label="v / m/s",
    vmin=0,
    vmax=1.5,
    title="Mean",
)
plt.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0, hspace=0.6)
plt.show()

Now the same for a specific measurement area:

In [ ]:
from pedpy import AxisAlignedMeasurementArea, SpeedMethod, compute_speed_profile

voronoi_speed_profile = compute_speed_profile(
    data=resorted_profile_data_measurement_area,
    walkable_area=walkable_area,
    axis_aligned_measurement_area=profile_measurement_area,
    grid_intersections_area=grid_cell_intersection_area_measurement_area,
    grid_size=grid_size,
    speed_method=SpeedMethod.VORONOI,
)

arithmetic_speed_profile = compute_speed_profile(
    data=resorted_profile_data_measurement_area,
    walkable_area=walkable_area,
    axis_aligned_measurement_area=profile_measurement_area,
    grid_intersections_area=grid_cell_intersection_area_measurement_area,
    grid_size=grid_size,
    speed_method=SpeedMethod.ARITHMETIC,
)

mean_speed_profile = compute_speed_profile(
    data=profile_data,
    walkable_area=walkable_area,
    axis_aligned_measurement_area=profile_measurement_area,
    grid_size=grid_size,
    speed_method=SpeedMethod.MEAN,
)

gauss_speed_profile = compute_speed_profile(
    data=profile_data,
    walkable_area=walkable_area,
    axis_aligned_measurement_area=profile_measurement_area,
    grid_size=grid_size,
    gaussian_width=0.5,
    speed_method=SpeedMethod.GAUSSIAN,
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_profiles

fig, ((ax0, ax1), (ax2, ax3)) = plt.subplots(nrows=2, ncols=2)
fig.suptitle("Speed profile")
cm = plot_profiles(
    walkable_area=walkable_area,
    measurement_area=profile_measurement_area,
    profiles=voronoi_speed_profile,
    axes=ax0,
    label="v / m/s",
    vmin=0,
    vmax=1.5,
    title="Voronoi",
)
cm = plot_profiles(
    walkable_area=walkable_area,
    measurement_area=profile_measurement_area,
    profiles=arithmetic_speed_profile,
    axes=ax1,
    label="v / m/s",
    vmin=0,
    vmax=1.5,
    title="Arithmetic",
)
cm = plot_profiles(
    walkable_area=walkable_area,
    measurement_area=profile_measurement_area,
    profiles=gauss_speed_profile,
    axes=ax2,
    label="v / m/s",
    vmin=0,
    vmax=1.5,
    title="Gauss",
)
cm = plot_profiles(
    walkable_area=walkable_area,
    measurement_area=profile_measurement_area,
    profiles=mean_speed_profile,
    axes=ax3,
    label="v / m/s",
    vmin=0,
    vmax=1.5,
    title="Mean",
)
plt.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0, hspace=0.6)
plt.show()

### Density Profiles

Currently, it is possible to compute either the Voronoi, classic or Gaussian density profiles.

**Voronoi density profile**

In each cell the Voronoi speed $v_{voronoi}$ is defined as

$$
    v_{voronoi}(t) = { \int\int v_{xy} dxdy \over A(M)},
$$ 
where $v_{xy} = v_i$ is the individual speed of each pedestrian, whose $V_i \cap M$ and $A(M)$ the area the grid cell.
 

**Classic density profile**

In each cell the density $\rho_{classic}$ is defined by 

$$  
    \rho_{classic} = {N \over A(M)},
$$
where $N$ is the number of pedestrians inside the grid cell $M$ and the area of that grid cell ($A(M)$).


**Gaussian density profile**

In each cell the density $\rho_{gaussian}$ is defined by 
    
$$    
    \rho_{gaussian} = \sum_{i=1}^{N}{\delta (\boldsymbol{r}_i - \boldsymbol{c})},
$$

where $\boldsymbol{r}_i$ is the position of a pedestrian and $\boldsymbol{c}$ is the center of the grid cell. Finally $\delta(x)$ is approximated by a Gaussian

$$    
    \delta(x) = \frac{1}{\sqrt{\pi}a}\exp[-x^2/a^2]
$$

In [ ]:
from pedpy import DensityMethod, compute_density_profile

# here it is important to use the resorted data, as it needs to be in the same ordering as "grid_cell_intersection_area"
voronoi_density_profile = compute_density_profile(
    data=resorted_profile_data,
    walkable_area=walkable_area,
    grid_intersections_area=grid_cell_intersection_area,
    grid_size=grid_size,
    density_method=DensityMethod.VORONOI,
)

# here the unsorted data can be used
classic_density_profile = compute_density_profile(
    data=profile_data,
    walkable_area=walkable_area,
    grid_size=grid_size,
    density_method=DensityMethod.CLASSIC,
)

gaussian_density_profile = compute_density_profile(
    data=profile_data,
    walkable_area=walkable_area,
    grid_size=grid_size,
    density_method=DensityMethod.GAUSSIAN,
    gaussian_width=0.5,
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_profiles

fig, (ax0, ax1, ax2) = plt.subplots(nrows=1, ncols=3, layout="constrained")
fig.set_size_inches(12, 5)
fig.suptitle("Density profile")
cm = plot_profiles(
    walkable_area=walkable_area,
    profiles=voronoi_density_profile,
    axes=ax0,
    label="$\\rho$ / 1/$m^2$",
    vmin=0,
    vmax=8,
    title="Voronoi",
)
cm = plot_profiles(
    walkable_area=walkable_area,
    profiles=classic_density_profile,
    axes=ax1,
    label="$\\rho$ / 1/$m^2$",
    vmin=0,
    vmax=8,
    title="Classic",
)
cm = plot_profiles(
    walkable_area=walkable_area,
    profiles=gaussian_density_profile,
    axes=ax2,
    label="$\\rho$ / 1/$m^2$",
    vmin=0,
    vmax=8,
    title="Gaussian",
)

plt.show()

Now the same for a specific measurement area:

In [ ]:
from pedpy import DensityMethod, compute_density_profile

# here it is important to use the resorted data, as it needs to be in the same ordering as "grid_cell_intersection_area"
voronoi_density_profile = compute_density_profile(
    data=resorted_profile_data_measurement_area,
    walkable_area=walkable_area,
    axis_aligned_measurement_area=profile_measurement_area,
    grid_intersections_area=grid_cell_intersection_area_measurement_area,
    grid_size=grid_size,
    density_method=DensityMethod.VORONOI,
)

# here the unsorted data can be used
classic_density_profile = compute_density_profile(
    data=profile_data,
    walkable_area=walkable_area,
    axis_aligned_measurement_area=profile_measurement_area,
    grid_size=grid_size,
    density_method=DensityMethod.CLASSIC,
)

gaussian_density_profile = compute_density_profile(
    data=profile_data,
    walkable_area=walkable_area,
    axis_aligned_measurement_area=profile_measurement_area,
    grid_size=grid_size,
    density_method=DensityMethod.GAUSSIAN,
    gaussian_width=0.5,
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_profiles

fig, (ax0, ax1, ax2) = plt.subplots(nrows=1, ncols=3, layout="constrained")
fig.set_size_inches(12, 5)
fig.suptitle("Density profile")
cm = plot_profiles(
    walkable_area=walkable_area,
    measurement_area=profile_measurement_area,
    profiles=voronoi_density_profile,
    axes=ax0,
    label="$\\rho$ / 1/$m^2$",
    vmin=0,
    vmax=8,
    title="Voronoi",
)
cm = plot_profiles(
    walkable_area=walkable_area,
    measurement_area=profile_measurement_area,
    profiles=classic_density_profile,
    axes=ax1,
    label="$\\rho$ / 1/$m^2$",
    vmin=0,
    vmax=8,
    title="Classic",
)
cm = plot_profiles(
    walkable_area=walkable_area,
    measurement_area=profile_measurement_area,
    profiles=gaussian_density_profile,
    axes=ax2,
    label="$\\rho$ / 1/$m^2$",
    vmin=0,
    vmax=8,
    title="Gaussian",
)

plt.show()

### Density and Speed Profiles

An other option is to compute both kinds of profile at the same time:

In [ ]:
from pedpy import compute_profiles

min_frame_profiles = 250  # We use here just an excerpt of the
max_frame_profiles = 400  # trajectory data to reduce compute time
grid_size = 0.4

density_profiles, speed_profiles = compute_profiles(
    data=pd.merge(
        individual_cutoff[individual_cutoff.frame.between(min_frame_profiles, max_frame_profiles)],
        individual_speed[individual_speed.frame.between(min_frame_profiles, max_frame_profiles)],
        on=[ID_COL, FRAME_COL],
    ),
    walkable_area=walkable_area.polygon,
    grid_size=grid_size,
    speed_method=SpeedMethod.ARITHMETIC,
    density_method=DensityMethod.VORONOI,
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_profiles

fig, (ax0, ax1) = plt.subplots(nrows=1, ncols=2)
cm = plot_profiles(
    walkable_area=walkable_area,
    profiles=density_profiles,
    axes=ax0,
    label="$\\rho$ / 1/$m^2$",
    vmin=0,
    vmax=12,
    title="Density",
)
cm = plot_profiles(
    walkable_area=walkable_area,
    profiles=speed_profiles,
    axes=ax1,
    label="v / m/s",
    vmin=0,
    vmax=2,
    title="Speed",
)
fig.tight_layout(pad=2)
plt.show()

Now the same for a specific measurement area:

In [ ]:
from pedpy import compute_profiles

min_frame_profiles = 250  # We use here just an excerpt of the
max_frame_profiles = 400  # trajectory data to reduce compute time
grid_size = 0.4

density_profiles, speed_profiles = compute_profiles(
    data=pd.merge(
        individual_cutoff[individual_cutoff.frame.between(min_frame_profiles, max_frame_profiles)],
        individual_speed[individual_speed.frame.between(min_frame_profiles, max_frame_profiles)],
        on=[ID_COL, FRAME_COL],
    ),
    walkable_area=walkable_area.polygon,
    axis_aligned_measurement_area=profile_measurement_area,
    grid_size=grid_size,
    speed_method=SpeedMethod.ARITHMETIC,
    density_method=DensityMethod.VORONOI,
)

In [ ]:
import matplotlib.pyplot as plt

from pedpy import plot_profiles

fig, (ax0, ax1) = plt.subplots(nrows=1, ncols=2)
cm = plot_profiles(
    walkable_area=walkable_area,
    measurement_area=profile_measurement_area,
    profiles=density_profiles,
    axes=ax0,
    label="$\\rho$ / 1/$m^2$",
    vmin=0,
    vmax=12,
    title="Density",
)
cm = plot_profiles(
    walkable_area=walkable_area,
    measurement_area=profile_measurement_area,
    profiles=speed_profiles,
    axes=ax1,
    label="v / m/s",
    vmin=0,
    vmax=2,
    title="Speed",
)
fig.tight_layout(pad=2)
plt.show()

## Spatial Analysis 
This section corresponds to analysis method which can be used to characterise different crowds or group formations.
These methods may include measurement of the time-to-collision, pair-distribution function and measurement of crowd polarization.


### Pair-distribution function (PDF)

This method is inspired from condensed matter description and used in the work of [Cordes et al. (2023)](https://doi.org/10.1093/pnasnexus/pgae120) following [Karamousas et al. (2014)](https://doi.org/10.1103/PhysRevLett.113.238701).
The pair-distribution function (PDF): 

$$g(r)=P(r)/P_{Ni}(r)$$.

"Quantifies the probability that two interacting pedestrians are found a given distance r apart, renormalized by the probability $P_{Ni}$ of measuring this distance for pedestrians that do not interact."

In this method, "interacting pedestrians" are defined as pedestrians that are present in the same spatial domain at the same time. One should also keep in mind that in its current implementation, the method does not take into account walls and corners, which should in theory block any "interaction" between pedestrians on opposite sides of the obstacles.

The probability $P_{Ni}$ is approximated here by time randomising the original trajectory file. For this randomisation process, only the frame numbers of the trajectory file are shuffled. The created "randomised trajectories" contain random pedestrian positions, composed only of positions present in the original trajectory file. This method helps account for pedestrians' preferred space utilisation, which can be due to terrain features or social behaviours. One should note that the number of positions selected for each frame is also random during the creation of the randomised trajectory file. The random process should ensure a uniform distribution of positions for each frame. However, to smooth any noise that this method may induce, we recommend using a higher `randomisation_stacking` number (see details in the next section).

The pair-distribution function of a given crowd recording can be computed using the following instructions:

In [ ]:
from pedpy import compute_pair_distribution_function

# Compute pair distribution function
radius_bins, pair_distribution = compute_pair_distribution_function(
    traj_data=traj, radius_bin_size=0.1, randomisation_stacking=1
)

In [ ]:
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(5, 5))
ax1.plot(radius_bins, pair_distribution)
ax1.set_title("Pair Distribution Function")
ax1.set_xlabel("$r$", fontsize=16)
ax1.set_ylabel("$g(r)$", fontsize=16)
ax1.grid(True, alpha=0.3)
plt.show()

#### Parameters of the PDF

The function `compute_pair_distribution_function` has two main parameters:

- `radius_bin_size` is the size of the radius bins for which probability will be computed. On one hand a larger bin size results in smoother pdf but decreases the accuracy of the description, as more individuals can be detected in each bin. On the other hand, a smaller bin will increase the accuracy of the description but may lead to noisy or `Nan` values as each bin may not be populated (leading to invalid divisions). We suggest using a bin size value between 0.1 and 0.3 m as these values are close to order of magniture of a chest depth.
- `randomisation_stacking` is the number of time the data stacked before being shuffled in order to compute the probability $P_{Ni}$ of measuring given pair-wise distances for pedestrians that do not interact. Stacking the data multiple times helps harmonize the random positions more effectively, ensuring that the PDF converges to results that are independent of the randomization method.

First we show the influence of varying `radius_bin_size` on the result:

In [ ]:
from pedpy import compute_pair_distribution_function

radius_bin_sizes = [0.05, 0.1, 0.25, 0.5]

varying_radius_bin_sizes = [
    (
        i,
        compute_pair_distribution_function(
            traj_data=traj,
            radius_bin_size=radius_bin_size,
            randomisation_stacking=1,
        ),
    )
    for i, radius_bin_size in enumerate(radius_bin_sizes)
]

And now how `randomisation_stacking` influences the result:

In [ ]:
from time import time

from pedpy import (
    compute_pair_distribution_function,
)

randomisation_stackings = [1, 3, 5]

varying_randomisation_stacking = []

for i, randomisation_stacking in enumerate(randomisation_stackings):
    begin_time = time()

    pdf = compute_pair_distribution_function(
        traj_data=traj,
        radius_bin_size=0.15,
        randomisation_stacking=randomisation_stacking,
    )
    end_time = time()

    varying_randomisation_stacking.append((i, pdf, end_time - begin_time))

These variations generate the following result:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.cm import twilight

cmap = twilight

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10, 5))
ax0.set_title("Effect of `radius_bin_size`")

for i, pdf in varying_radius_bin_sizes:
    radius_bins, pair_distribution = pdf
    ax0.plot(
        radius_bins,
        pair_distribution,
        color=twilight(i / len(varying_radius_bin_sizes)),
        label="$r_{bin}=$" + str(radius_bin_sizes[i]),
    )

ax0.set_ylim((0, 1.3))
ax0.set_xlabel("$r$", fontsize=16)
ax0.set_ylabel("$g(r)$", fontsize=16)
ax0.grid(True, alpha=0.3)
ax0.legend(title="Bin sizes")

ax1.set_title("Effect of 'randomisation_stacking'")
for i, pdf, _time in varying_randomisation_stacking:
    radius_bins, pair_distribution = pdf
    ax1.plot(
        radius_bins,
        pair_distribution,
        color=cmap(i / len(varying_randomisation_stacking)),
        label=str(randomisation_stackings[i]) + " times: " + str(np.round(_time, 2)) + "s",
    )
ax1.set_ylim((0, 1.3))
ax1.set_ylabel("$g(r)$", fontsize=16)
ax1.set_xlabel("$r$", fontsize=16)
ax1.grid(True, alpha=0.3)

fig.tight_layout()
ax1.legend(title="Nb of stacks: Execution time")
plt.show()